# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 1 и 1,33 потому что первое проверка на нан и далее сумма в серии, во втором вместо нан ставим 0 и все делим на 3 получим 1,33
2. Ответ 2, потому что вначаде мы убираем пробелы и приводим в ниж.регистр дали понимаем что уникальных тут 2 символа
3. Ответ 3, так как вопрос с подвохом тут мы не меням датафрейм а проверяем что было бы если...
4. Ответ 1005, потому что мы имеем две строки с них убираем пробелы и превращаем в цифры далее выводим сумму этих цифр
5. Ответ [9, 10]. Потому что вначале мы имеем сарию из строк-дат, после чего превращаем в таймштамп и на принте выводим список из всех месяцев в серии

In [25]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [26]:
# 1
import pandas as pd

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

def clean(path):
    df = pd.read_csv(path)
    print(f'Исходных строк {len(df)}')
    print('-' * 40)

    # amount to float
    df['amount'] = pd.to_numeric(df['amount'].str.replace(' ', ''))
    print(f'Строк после сумм во флоат {len(df)}')
    print('-' * 40)

    # city through dict
    df['city'] = df['city'].str.strip().str.lower().map(CITY)
    print(f'Строк после редактирования городов {len(df)}')
    print('-' * 40)

    # Убираем дубли
    df = df.drop_duplicates()
    print(f'Строк после удаления дублей {len(df)}')
    print('-' * 40)

    # строку в даты
    df['order_date'] = pd.to_datetime(df['order_date'])
    print(f'Строк после превращения в даты {len(df)}')
    print('-' * 40)


    # bez TEST
    df = df.loc[df['customer_id'] != 'TEST']
    print(f'Строк после убирания TEST {len(df)}')
    print('-' * 40)


    assert df['amount'].isna().sum() == 0
    assert df['city'].isna().sum() == 0
    assert df['order_id'].is_unique

    return df


    



df = clean('orders.csv')

Исходных строк 5463
----------------------------------------
Строк после сумм во флоат 5463
----------------------------------------
Строк после редактирования городов 5463
----------------------------------------
Строк после удаления дублей 5283
----------------------------------------
Строк после превращения в даты 5283
----------------------------------------
Строк после убирания TEST 5280
----------------------------------------


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [27]:
# 2
paid = df['status'] == 'оплачен'
sep = df['order_date'].dt.month == 9
octo = df['order_date'].dt.month == 10

rev_sep = df.loc[(paid & sep), 'amount'].sum()
rev_octo = df.loc[(paid & octo), 'amount'].sum()

perc_change = ((rev_octo - rev_sep) / rev_sep) * 100
print(f'Общая выручка за сентябрь {rev_sep} | Общая выручка за октябрь {rev_octo}')
print(f'Изменине по всем городам {perc_change:.1f}%')
print('-' * 50)

changes = {}
for city in df['city'].unique():
    curr_city = df['city'] == city
    curr_rev_sep = df.loc[(paid & sep & curr_city), 'amount'].sum()
    curr_rev_octo = df.loc[(paid & octo & curr_city), 'amount'].sum()

    curr_city_per_change = ((curr_rev_octo - curr_rev_sep) / curr_rev_sep) * 100
    changes[city] = curr_city_per_change

    print(f'Выручка {city} за сентябрь {curr_rev_sep} | Выручка {city} за октябрь {curr_rev_octo}')
    print(f'Изменение города {city} {curr_city_per_change:.1f}%')
    print('-' * 50)

worst_change = min(changes, key=changes.get)
print(f'Худшее измение в процентах {worst_change} {changes[worst_change]:.1f}%')


Общая выручка за сентябрь 35243410.0 | Общая выручка за октябрь 34419520.0
Изменине по всем городам -2.3%
--------------------------------------------------
Выручка Алматы за сентябрь 14501070.0 | Выручка Алматы за октябрь 14643020.0
Изменение города Алматы 1.0%
--------------------------------------------------
Выручка Астана за сентябрь 9034220.0 | Выручка Астана за октябрь 8850530.0
Изменение города Астана -2.0%
--------------------------------------------------
Выручка Караганда за сентябрь 3648520.0 | Выручка Караганда за октябрь 3371680.0
Изменение города Караганда -7.6%
--------------------------------------------------
Выручка Шымкент за сентябрь 4558650.0 | Выручка Шымкент за октябрь 4044010.0
Изменение города Шымкент -11.3%
--------------------------------------------------
Выручка Атырау за сентябрь 923990.0 | Выручка Атырау за октябрь 969860.0
Изменение города Атырау 5.0%
--------------------------------------------------
Выручка Актобе за сентябрь 2576960.0 | Выручка Актоб

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

| Шаг | Убрано строк | Изменение выручки |
| :--- | ---: | ---: |
| Исходные данные | — | −22,7% |
| Удаление дублей | 180 | −19,6% |
| Удаление TEST | 3 | −2,3% |

Сильнее всего исказил цифру ТЕСТ, там заоблачные суммы

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [28]:
print(df.isna().sum())

order_id           0
order_date         0
customer_id        0
city               0
category           0
amount             0
delivery_days    158
status             0
dtype: int64


In [29]:
print(df.loc[df['delivery_days'].isna(), ['city']].value_counts())

city     
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64


In [30]:
print(df.loc[df['delivery_days'].isna(), ['city', 'status']].value_counts())

city       status 
Алматы     оплачен    61
Астана     оплачен    35
Караганда  оплачен    23
Шымкент    оплачен    17
Астана     отменён     5
Шымкент    отменён     5
Актобе     оплачен     4
Атырау     оплачен     4
Алматы     отменён     2
Атырау     отменён     1
Караганда  отменён     1
Name: count, dtype: int64


In [31]:
# 4
sh = df[df["city"] == "Шымкент"]

# (а) dropna
share_dropna = (sh["delivery_days"].dropna() <= 2).mean() * 100

# (б) fillna(медиана)
med_all = df["delivery_days"].median()
med_sh = sh["delivery_days"].median()
share_med_all = (sh["delivery_days"].fillna(med_all) <= 2).mean() * 100
share_med_sh = (sh["delivery_days"].fillna(med_sh) <= 2).mean() * 100

# (в) fillna(0)
share_zero = (sh["delivery_days"].fillna(0) <= 2).mean() * 100

print(f"(а) dropna:               {share_dropna:.2f}%")
print(f"(б) fillna(общ. медиана): {share_med_all:.2f}%")
print(f"(б) fillna(мед. Шымкент): {share_med_sh:.2f}%")
print(f"(в) fillna(0):            {share_zero:.2f}%")

(а) dropna:               29.84%
(б) fillna(общ. медиана): 31.83%
(б) fillna(мед. Шымкент): 28.99%
(в) fillna(0):            31.83%


Я считаю что наиболее приблbженные данные к истине это заполнение по медиане города, так как там базируются локальные данные, также мы не можем абсолютно убрать неизвестную дату доставки так как это почти 10 процентов от всех заказов в городе, и не можем прировнять их к нулю так как увеличим долю заказов в статистике

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [36]:
# 5

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}
df_4 = pd.read_csv("orders.csv")
df_4 = df_4.drop_duplicates()
df_4["city"] = df_4["city"].str.strip().str.lower().str.replace(' ', '').map(CITY)
print(df["city"].nunique(), len(df_4))

6 5283


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [33]:
# 6
df_task_6 = clean('orders.csv')

days = df_task_6['order_date'].dt.day_name().value_counts()
print(days)

diff_of_days = ((df_task_6['order_date'].dt.day_name().value_counts().min() / df_task_6['order_date'].dt.day_name().value_counts().max())-1) * 100
print(f'Самый загруженый день {days.idxmax()}-786 заказа, Самый лекгий день {days.idxmin()}-684 заказа, Разница между ними в {diff_of_days:.2f}%')

Исходных строк 5463
----------------------------------------
Строк после сумм во флоат 5463
----------------------------------------
Строк после редактирования городов 5463
----------------------------------------
Строк после удаления дублей 5283
----------------------------------------
Строк после превращения в даты 5283
----------------------------------------
Строк после убирания TEST 5280
----------------------------------------
order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
Самый загруженый день Thursday-786 заказа, Самый лекгий день Monday-684 заказа, Разница между ними в -12.98%


Самый загруженый день Thursday-786 заказа, Самый лекгий день Monday-684 заказа, Разница между ними в -12.98%

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [34]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [35]:
# 8